# CropDetector — Phase 9, Part 5: INT8 quantization of the Stage A input validator

**Standalone Colab notebook.** This notebook uses two previously saved archives:

- `CropDetector_Phase9_Part2_Data.zip` — verified manifests (not photographs).
- `CropDetector_Phase9_Part3_Results.zip` — frozen MobileNetV2-0.35 trained Stage A model.

You do **not** need to rerun any training notebooks, and the original 15-class disease classifier remains unchanged.

### Decision carried forward from the independently evaluated Part 4
- **Default provisional plant threshold: 0.50** (`plant_score = sigmoid(-nonplant_logit)`).
- The original **0.999902** operating threshold is stored only for comparison. It rejected many independent flower images.
- Stage A tests plant-like versus non-plant. It **cannot** identify the four supported crops or guarantee that a photograph is diagnostic-quality.
- The independent Part 4 data are **not** used to fit quantization scales, select a threshold, or train a new network.

**Deliverable:** one downloadable ZIP containing FP32 and INT8 LiteRT models, an INT8 quantization configuration, development regression measurements, and golden test tensors for later Android parity testing.

## Cell 1 — Install packages and initialize the notebook
Colab already includes TensorFlow and Pillow. The only new package is `kagglehub`, for restoring the original training-photo source.

In [ ]:
%pip -q install kagglehub

import os, io, json, time, gc, hashlib, shutil, subprocess
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from scipy.special import expit
from IPython.display import display
import tensorflow as tf
import kagglehub
from google.colab import files

SEED = 42
IMG_SIZE = (160, 160)
ROOT = Path('/content/CropDetector_Phase9_Part5')
EXPORT = ROOT / 'export'
ROOT.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Python runtime ready')

## Cell 2 — Upload the previously saved Part 2 and Part 3 ZIPs
Both archives must be the exact originals downloaded in earlier phases. SHA-256 checks ensure that we reuse the right manifest and classifier.

In [ ]:
def get_zip(filename):
    destination = ROOT / filename
    if not destination.exists():
        print('Upload:', filename)
        uploaded = files.upload()
        assert filename in uploaded, f'Expected {filename}, received {list(uploaded)}'
        destination.write_bytes(uploaded[filename])
    return destination

PART2_ZIP = get_zip('CropDetector_Phase9_Part2_Data.zip')
PART3_ZIP = get_zip('CropDetector_Phase9_Part3_Results.zip')

manifest_sha = hashlib.sha256(PART2_ZIP.read_bytes()).hexdigest()
with ZipFile(PART2_ZIP) as archive:
    audit = json.loads(archive.read('audit_report.json'))
    train_df = pd.read_csv(io.BytesIO(archive.read('validator_train.csv')))
    dev_df = pd.read_csv(io.BytesIO(archive.read('validator_dev.csv')))
with ZipFile(PART3_ZIP) as archive:
    stage_config = json.loads(archive.read('stageA_config.json'))
    model_bytes = archive.read(stage_config['model_file'])

assert audit['phase'] == 'Phase9_Part2'
assert stage_config['phase'] == 'Phase9_Part3_StageA'
assert stage_config['source_manifest_sha256'] == manifest_sha, 'Wrong Part 2 manifest archive'
assert hashlib.sha256(model_bytes).hexdigest() == stage_config['model_sha256'], 'Model checksum mismatch'
assert len(train_df) == 4065 and len(dev_df) == 1012
assert set(train_df['Group']).isdisjoint(set(dev_df['Group']))
assert set(train_df['SHA256']).isdisjoint(set(dev_df['SHA256']))
assert set(train_df['Target'].unique()) == {0,1}

MODEL_PATH = ROOT / 'crop_validator_stageA.keras'
MODEL_PATH.write_bytes(model_bytes)
print('Verified Part 2:', manifest_sha)
print('Verified Stage A model:', stage_config['model_sha256'])
print('Train / dev images:', len(train_df), '/', len(dev_df))
display(train_df.groupby(['Source','Target']).size().to_frame('Count'))

## Cell 3 — Restore the exact original photo sources
This is **not training**. The images are needed only for representative INT8 calibration and a development regression check. Source paths and image fingerprints are reused from the Part 2 manifest.

In [ ]:
PLANTDOC_ROOT = ROOT / 'PlantDoc-Dataset'
COMMIT = audit['plantdoc_git_commit']
REPO = 'https://github.com/pratikkayal/PlantDoc-Dataset.git'
if not (PLANTDOC_ROOT / '.git').exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(PLANTDOC_ROOT)], check=True)
head = subprocess.check_output(['git','-C',str(PLANTDOC_ROOT),'rev-parse','HEAD'], text=True).strip()
if head != COMMIT:
    subprocess.run(['git','-C',str(PLANTDOC_ROOT),'fetch','--depth','1','origin',COMMIT],check=True)
    subprocess.run(['git','-C',str(PLANTDOC_ROOT),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(PLANTDOC_ROOT),'rev-parse','HEAD'],text=True).strip() == COMMIT

NATURAL_ROOT = Path(kagglehub.dataset_download('prasunroy/natural-images'))

def resolve_path(row):
    root = PLANTDOC_ROOT if row.Source == 'PlantDoc' else NATURAL_ROOT
    return str(root / str(row.RelativePath))

for frame in (train_df, dev_df):
    frame['ResolvedPath'] = frame.apply(resolve_path, axis=1)
    missing = frame.loc[~frame.ResolvedPath.map(lambda p:Path(p).is_file())]
    assert missing.empty, f'Missing source files: {missing[["Source","RelativePath"]].head().to_dict("records")}'

print('PlantDoc commit:', COMMIT)
print('Natural Images root:', NATURAL_ROOT)
print('All referenced training and development photos exist')

## Cell 4 — Select representative images from **training only**
Calibration uses 150 plant-like and 150 non-plant training images. We verify every selected image against its manifest hash. Development/test images do not determine the quantization scales.

In [ ]:
rng = np.random.default_rng(SEED)
plant_rep = train_df[train_df.Target == 0].sample(n=150,random_state=SEED)
object_rep = train_df[train_df.Target == 1].sample(n=150,random_state=SEED)
rep_df = pd.concat([plant_rep, object_rep], ignore_index=True)
rep_df = rep_df.iloc[rng.permutation(len(rep_df))].reset_index(drop=True)

# Verify all representative files (these are the only images used for PTQ calibration).
def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

for row in rep_df.itertuples(index=False):
    assert file_sha(row.ResolvedPath).lower() == str(row.SHA256).lower(), row.RelativePath


def load_rgb160(path):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert('RGB')
        image = image.resize((160,160), resample=Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.float32)


def representative_dataset():
    for path in rep_df.ResolvedPath:
        yield [load_rgb160(path)[None,...]]

print('Representative images:',len(rep_df))
print(rep_df.groupby(['Source','Target']).size())

## Cell 5 — Load the frozen model and export FP32 + fully integer INT8
The trained `.keras` file produces **one non-plant logit**. Higher logits favor non-plant. Dropout is automatically inactive during conversion/inference. Input pixels remain RGB float32 in the range 0–255 before any integer-tensor quantization.

In [ ]:
source_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
assert tuple(source_model.input_shape[1:]) == (160,160,3)
assert int(source_model.output_shape[-1]) == 1
source_model.trainable = False

fp32_converter = tf.lite.TFLiteConverter.from_keras_model(source_model)
fp32_bytes = fp32_converter.convert()
FP32_PATH = EXPORT / 'crop_validator_stageA_fp32.tflite'
FP32_PATH.write_bytes(fp32_bytes)

int8_converter = tf.lite.TFLiteConverter.from_keras_model(source_model)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_dataset
int8_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8
int8_bytes = int8_converter.convert()
INT8_PATH = EXPORT / 'crop_validator_stageA_int8.tflite'
INT8_PATH.write_bytes(int8_bytes)

print('FP32 size: %.3f MiB' % (FP32_PATH.stat().st_size / 1048576))
print('INT8 size: %.3f MiB' % (INT8_PATH.stat().st_size / 1048576))
print('Storage reduction: %.2f%%' % (100*(1-INT8_PATH.stat().st_size/FP32_PATH.stat().st_size)))

## Cell 6 — Verify tensor types, dimensions, and quantization parameters
The INT8 version must use `int8` for both input and output, not silently fall back to floating-point tensors.

In [ ]:
def make_interpreter(path):
    interpreter = tf.lite.Interpreter(model_path=str(path), num_threads=2)
    interpreter.allocate_tensors()
    return interpreter

interpreters = {
    'FP32': make_interpreter(FP32_PATH),
    'INT8': make_interpreter(INT8_PATH)
}

spec_rows = []
for name, interpreter in interpreters.items():
    in_info = interpreter.get_input_details()[0]
    out_info = interpreter.get_output_details()[0]
    assert tuple(in_info['shape']) == (1,160,160,3)
    assert tuple(out_info['shape']) == (1,1)
    if name=='INT8':
        assert in_info['dtype'] == np.int8 and out_info['dtype'] == np.int8
        assert in_info['quantization'][0] > 0 and out_info['quantization'][0] > 0
    spec_rows.append({
        'Model':name,
        'Input dtype':np.dtype(in_info['dtype']).name,
        'Output dtype':np.dtype(out_info['dtype']).name,
        'Input scale':float(in_info['quantization'][0]),
        'Input zero point':int(in_info['quantization'][1]),
        'Output scale':float(out_info['quantization'][0]),
        'Output zero point':int(out_info['quantization'][1]),
    })
spec_df = pd.DataFrame(spec_rows)
display(spec_df)
spec_df.to_csv(EXPORT/'tensor_specs.csv',index=False)

## Cell 7 — Numerically correct TFLite inference (FP32 or INT8)
All input/output scale factors are read from the actual model tensors. Do not hardcode the main disease classifier's INT8 scale factors: Stage A has a different model.

In [ ]:
def tflite_logit(interpreter, image_float):
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    value = np.asarray(image_float,dtype=np.float32).reshape((1,160,160,3))
    if np.issubdtype(inp['dtype'],np.integer):
        scale,zero = inp['quantization']
        bounds = np.iinfo(inp['dtype'])
        value = np.clip(np.rint(value/scale)+zero,bounds.min,bounds.max).astype(inp['dtype'])
    else:
        value = value.astype(inp['dtype'])
    interpreter.set_tensor(inp['index'],value)
    interpreter.invoke()
    result = interpreter.get_tensor(out['index']).reshape(-1)[0]
    if np.issubdtype(out['dtype'],np.integer):
        scale,zero = out['quantization']
        result = (float(result)-zero)*scale
    return float(result)

trial = load_rgb160(dev_df.iloc[0].ResolvedPath)
keras_logit = float(np.asarray(source_model(trial[None,...],training=False)).reshape(-1)[0])
fp32_logit = tflite_logit(interpreters['FP32'],trial)
int8_logit = tflite_logit(interpreters['INT8'],trial)
print('Single-image check (non-plant logits):')
print('Keras: %.6f, FP32: %.6f, INT8: %.6f'%(keras_logit, fp32_logit, int8_logit))

## Cell 8 — Regression test on all 1,012 Stage A development images
This reuses the **previous development split**, so it is a *quantization regression test*, not a newly independent accuracy estimate. The threshold 0.5 stays fixed; the strict threshold from Part 3 is shown only as a diagnostic.

In [ ]:
model_logits = {'FP32':[], 'INT8':[]}
for idx, row in enumerate(dev_df.itertuples(index=False),1):
    image = load_rgb160(row.ResolvedPath)
    for name, interpreter in interpreters.items():
        model_logits[name].append(tflite_logit(interpreter,image))
    if idx % 200 == 0:
        print('Evaluated:',idx,'/',len(dev_df))

for name in model_logits:
    model_logits[name] = np.asarray(model_logits[name],dtype=np.float64)

true = dev_df['Target'].to_numpy(dtype=np.int32)
thresholds = {'Default 0.5':0.5,'Part 3 strict (diagnostic)':float(stage_config['accept_if_plant_score_at_least'])}
records=[]
for name,logits in model_logits.items():
    plant_score=expit(-logits)
    for rule,t in thresholds.items():
        accepted=plant_score >= t
        plant=(true==0)
        obj=(true==1)
        records.append({
            'Model':name,'Rule':rule,'Threshold':t,
            'Plant images':int(plant.sum()),
            'Plant retention (%)':100*accepted[plant].mean(),
            'Non-plant images':int(obj.sum()),
            'Non-plant rejection (%)':100*(~accepted[obj]).mean(),
            'Balanced accuracy (%)':50*(accepted[plant].mean()+(~accepted[obj]).mean()),
        })

regression=pd.DataFrame(records)
display(regression.round(5))
regression.to_csv(EXPORT/'quantization_dev_regression.csv',index=False)

pred_fp32=(model_logits['FP32']>=0).astype(int)
pred_int8=(model_logits['INT8']>=0).astype(int)
parity={
    'samples':len(dev_df),
    'fp32_int8_top1_agreement_percent':float(100*np.mean(pred_fp32==pred_int8)),
    'logit_mean_abs_error':float(np.mean(np.abs(model_logits['FP32']-model_logits['INT8']))),
    'logit_max_abs_error':float(np.max(np.abs(model_logits['FP32']-model_logits['INT8']))),
    'changed_decisions':int(np.sum(pred_fp32!=pred_int8)),
}
print('FP32 vs INT8 regression:',json.dumps(parity,indent=2))
pd.DataFrame({
    'Source':dev_df.Source,'Role':dev_df.Role,'Folder':dev_df.Folder,
    'Target':true,'FP32_nonplant_logit':model_logits['FP32'],
    'INT8_nonplant_logit':model_logits['INT8'],
    'FP32_plant_score':expit(-model_logits['FP32']),
    'INT8_plant_score':expit(-model_logits['INT8']),
}).to_csv(EXPORT/'development_image_scores.csv',index=False)

## Cell 9 — Check performance separately for supported leaves, unsupported leaves and objects
A general plant detector **should retain unsupported crops**, not reject them. Stage B will determine whether they are among the four recognized crop species.

In [ ]:
role_rows=[]
for name,logits in model_logits.items():
    score=expit(-logits)
    accept=score>=0.5
    for role,frame in dev_df.groupby('Role',sort=True):
        indices=frame.index.to_numpy()
        role_rows.append({
            'Model':name,'Role':role,'Images':len(indices),
            'Accepted as plant (%)':100*accept[indices].mean(),
            'Rejected (%)':100*(~accept[indices]).mean()
        })
role_df=pd.DataFrame(role_rows)
display(role_df.round(3))
role_df.to_csv(EXPORT/'development_by_role.csv',index=False)

## Cell 10 — Export 8 golden parity inputs for the ARM phone
Each `.bin` file contains 160 × 160 × 3 little-endian FP32 pixels **after PIL resizing**. Android should feed these tensors directly, converting to INT8 according to tensor metadata. Camera preprocessing is a separate test.

In [ ]:
GOLD = EXPORT/'golden_inputs'
GOLD.mkdir(exist_ok=True)
plant_cases=(dev_df.loc[dev_df.Target==0].groupby('CropLabel',sort=True)
             .head(1).head(4))
object_cases=(dev_df.loc[dev_df.Target==1].groupby('Folder',sort=True)
              .head(1).head(4))
cases=pd.concat([plant_cases,object_cases],ignore_index=True)
assert len(cases)==8

golden=[]
for i,row in enumerate(cases.itertuples(index=False)):
    image=load_rgb160(row.ResolvedPath)
    fname=f'golden_inputs/validator_{i:02d}.bin'
    np.asarray(image,dtype='<f4').tofile(EXPORT/fname)
    golden.append({
        'case':i,'source':row.Source,'role':row.Role,'folder':row.Folder,
        'expected_truth':int(row.Target),'input_file':fname,
        'input_shape':[1,160,160,3], 'input_dtype':'float32_little_endian',
        'expected_nonplant_logits':{
            name:tflite_logit(interpreter,image)
            for name,interpreter in interpreters.items()
        }
    })
with open(EXPORT/'golden_cases.json','w') as f:
    json.dump(golden,f,indent=2)
print('Golden parity cases saved:',len(golden))
print('Models:',list(interpreters))

## Cell 11 — Save deployment metadata, hashes and quantization results
**Do not use the Stage C disease classifier's temperature scaling or crop masks on this Stage A logit.** Stage A uses `plant_score = sigmoid(-nonplant_logit)` and provisional threshold 0.5.

In [ ]:
models_config={}
for name,path in [('FP32',FP32_PATH),('INT8',INT8_PATH)]:
    interpreter=interpreters[name]
    in_detail=interpreter.get_input_details()[0]
    out_detail=interpreter.get_output_details()[0]
    models_config[name]={
        'filename':path.name,
        'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
        'size_bytes':int(path.stat().st_size),
        'input_dtype':np.dtype(in_detail['dtype']).name,
        'output_dtype':np.dtype(out_detail['dtype']).name,
        'input_quantization':{
            'scale':float(in_detail['quantization'][0]),
            'zero_point':int(in_detail['quantization'][1])
        },
        'output_quantization':{
            'scale':float(out_detail['quantization'][0]),
            'zero_point':int(out_detail['quantization'][1])
        }
    }

config={
    'phase':'Phase9_Part5_StageA_INT8',
    'task':'plant-like-versus-non-plant',
    'input_shape':[1,160,160,3],
    'rgb_input_range':[0,255],
    'preprocessing':'EXIF transpose; RGB; bilinear resize; model rescales internally',
    'output':'one NON-PLANT logit',
    'plant_score_formula':'sigmoid(-nonplant_logit)',
    'provisional_plant_acceptance_threshold':0.5,
    'diagnostic_original_strict_threshold':float(stage_config['accept_if_plant_score_at_least']),
    'quantization_representative_images':len(rep_df),
    'quantization_source':'Stage A training only; 150 PlantDoc plus 150 Natural Images',
    'prior_stageA_model_sha256':stage_config['model_sha256'],
    'part2_manifest_sha256':manifest_sha,
    'models':models_config,
    'quantization_regression':parity,
    'independent_field_validation':False,
    'warning':'Plant-like detection only. Does not identify supported crops or image diagnostic suitability.'
}
with open(EXPORT/'stageA_deployment_config.json','w') as f:
    json.dump(config,f,indent=2)
with open(EXPORT/'README.txt','w') as f:
    f.write('CropDetector Phase 9 Part 5 — Stage A plant-like validator\n'
            'This is a research prototype. Stage A does not identify supported crops.\n'
            'Use provisional threshold plant_score >= 0.5; plant_score=sigmoid(-nonplant_logit).\n'
            'For INT8, obey quantization scales/zero-points in the model tensor metadata.\n'
            'Golden inputs are preprocessed little-endian float32 RGB arrays, 160x160x3.\n')

print('INT8 deployment config:')
print(json.dumps(config,indent=2)[:4800])

## Cell 12 — Download the complete portable ZIP
It contains both converted models, the Stage A operating configuration, evaluation tables and golden test tensors. Save it on your Mac; Colab's `/content` directory is temporary.

In [ ]:
archive=shutil.make_archive(
    str(ROOT/'CropDetector_Phase9_Part5_Results'),'zip',EXPORT
)
print('Created:',archive)
with ZipFile(archive) as z:
    for item in z.infolist():
        print(f'{item.file_size/1048576:6.3f} MiB  {item.filename}')
files.download(archive)

---
### After running this notebook
Upload `CropDetector_Phase9_Part5_Results.zip` here. We will review **the actual INT8 model size**, the FP32↔INT8 decision agreement, development plant retention, and non-plant rejection. Then we can integrate Stage A into Android and separately implement Stage B **Maize / Grape / Potato / Tomato / Other Crop** recognition.

**Remaining limitations:** Stage A was trained with PlantDoc positives and Natural Images negatives, so dataset-source bias remains possible. Part 4 independent tests are useful but not a fully untouched smartphone field test. Quantization does not solve crop-species rejection.